# Stages 04 & 05 — Spare Parts Orders & Sales EDA
**Dashboard page:** Spare Parts Analysis
**Tabs:** Orders EDA (Stage 4) · Sales EDA (Stage 5) · Inventory Position (Module 3)

**Dealer scoping rule:** Only customers whose code matches `Dealers.xlsx` are included.
**Document type:** Sales Document starting with 4 = purchase order; 6 = return.

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
orders = load("orders_clean.parquet")
sales  = load("sales_clean.parquet")
reject = load("orders_rejection_log.parquet") if (INTERIM/"orders_rejection_log.parquet").exists() else pd.DataFrame()

print(f"Orders clean : {len(orders):,} rows | categories: {dict(orders['mc_category'].value_counts()) if 'mc_category' in orders.columns else 'N/A'}")
print(f"Sales clean  : {len(sales):,} rows  | bill_class: {dict(sales['bill_class'].value_counts()) if 'bill_class' in sales.columns else 'N/A'}")
print(f"Rejection log: {len(reject):,} rows")


## Tab 1 — Orders EDA (Stage 4)

In [ ]:
sp = orders[orders.get("mc_category","").eq("Spare Parts") if "mc_category" in orders.columns else orders.index.notna()]
sp = sp.copy()

# Date column
date_col = next((c for c in ["Created On","Document Date","created_on"] if c in sp.columns),None)
if date_col: sp["month"] = pd.to_datetime(sp[date_col],errors="coerce").dt.to_period("M")

qty_col  = "Order Quantity (Item)" if "Order Quantity (Item)" in sp.columns else None
cqty_col = "Confirmed Quantity (Item)" if "Confirmed Quantity (Item)" in sp.columns else None
val_col  = "Net Value (Item)" if "Net Value (Item)" in sp.columns else None
lt_col   = "lead_time_days" if "lead_time_days" in sp.columns else None
fr_col   = "fill_rate" if "fill_rate" in sp.columns else None

print(f"Spare parts orders: {len(sp):,}")
if qty_col:  print(f"Total order qty  : {sp[qty_col].sum():,.0f}")
if cqty_col: print(f"Total confirmed  : {sp[cqty_col].sum():,.0f}")


In [ ]:
fig,axes = plt.subplots(2,2,figsize=(14,10))

# Monthly order value
if date_col and val_col:
    mv = sp.groupby("month")[val_col].sum().sort_index()
    mv.index = mv.index.astype(str)
    mv.plot(ax=axes[0,0],color=PALETTE[0],lw=2.5)
    axes[0,0].set_title("Monthly Spare Parts Order Value (LKR)")
    axes[0,0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x/1e6:.0f}M"))
    axes[0,0].tick_params(axis="x",rotation=45)

# Category mix
if "mc_category" in orders.columns:
    cat_cnt = orders["mc_category"].value_counts()
    cat_cnt.plot(kind="bar",ax=axes[0,1],color=PALETTE[:len(cat_cnt)],edgecolor="white")
    axes[0,1].set_title("Order Volume by Material Category")
    axes[0,1].tick_params(axis="x",rotation=30)
    for bar,val in zip(axes[0,1].patches,cat_cnt.values):
        axes[0,1].text(bar.get_x()+bar.get_width()/2,bar.get_height()+50,f"{val:,}",ha="center",fontsize=8)

# Fill rate distribution
if fr_col:
    sp[fr_col].clip(0,1).hist(bins=50,ax=axes[1,0],color=PALETTE[2],edgecolor="white",alpha=0.8)
    axes[1,0].axvline(sp[fr_col].mean(),color=PALETTE[1],ls="--",lw=2,label=f"Mean={sp[fr_col].mean():.2%}")
    axes[1,0].set_title("Fill Rate Distribution (confirmed/ordered)")
    axes[1,0].set_xlabel("Fill rate"); axes[1,0].legend()

# Lead time distribution
if lt_col:
    lt = sp[lt_col].clip(0,90)
    lt.hist(bins=60,ax=axes[1,1],color=PALETTE[3],edgecolor="white",alpha=0.8)
    axes[1,1].axvline(lt.mean(),color=PALETTE[1],ls="--",lw=2,label=f"Mean={lt.mean():.1f}d")
    axes[1,1].axvline(90,color=PALETTE[0],ls=":",lw=1.5,label="Lead time target=90d")
    axes[1,1].set_title("Lead Time Distribution (Good Issue Date - Created On)")
    axes[1,1].set_xlabel("Days"); axes[1,1].legend()

plt.tight_layout(); plt.show()


In [ ]:
# Rejection log
if len(reject)>0:
    print(f"Rejection log — fully short-shipped lines: {len(reject):,}")
    print(reject.head(10).to_string())
    if "mc_category" in reject.columns:
        print("
Rejections by category:")
        print(reject["mc_category"].value_counts().to_string())
else:
    print("No rejection log or zero rejected lines.")


## Tab 2 — Sales EDA (Stage 5)

In [ ]:
# bill_class: 'F2' = invoice (sale), 'RE'/'S1' = return
sale_mask = sales["bill_class"].isin(["F2","Invoice","sale","SALE"]) if "bill_class" in sales.columns else pd.Series([True]*len(sales))
ret_mask  = ~sale_mask

net_col   = "Net Sales" if "Net Sales" in sales.columns else None
qty_col   = "SlsVolQty" if "SlsVolQty" in sales.columns else None
prov_col  = "Province"  if "Province"  in sales.columns else None
dealer_col= "Dealer Code" if "Dealer Code" in sales.columns else None

if net_col:
    total_sales = sales.loc[sale_mask,net_col].sum()
    total_ret   = sales.loc[ret_mask, net_col].sum()
    print(f"Total net sales : {fmt_lkr(total_sales)}")
    print(f"Total returns   : {fmt_lkr(abs(total_ret))}")
    print(f"Net revenue     : {fmt_lkr(total_sales+total_ret)}")

if "dealer_type" in sales.columns:
    print("
Dealer type breakdown:")
    print(sales["dealer_type"].value_counts().to_string())


In [ ]:
fig,axes = plt.subplots(1,2,figsize=(14,5))

# Province breakdown
if prov_col and net_col:
    prov = sales.loc[sale_mask].groupby(prov_col)[net_col].sum().sort_values(ascending=False)
    prov.plot(kind="barh",ax=axes[0],color=PALETTE[0],edgecolor="white")
    axes[0].set_title("Net Sales by Province (LKR)")
    axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x/1e9:.1f}B"))

# Top dealers
if dealer_col and net_col:
    top_dlr = sales.loc[sale_mask].groupby(dealer_col)[net_col].sum().sort_values(ascending=False).head(15)
    top_dlr.sort_values().plot(kind="barh",ax=axes[1],color=PALETTE[2],edgecolor="white")
    axes[1].set_title("Top 15 Dealers by Net Sales (LKR)")
    axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x/1e6:.0f}M"))
plt.tight_layout(); plt.show()

# Monthly trend
date_col2 = next((c for c in ["Billing Date","Year_Month_str"] if c in sales.columns),None)
if date_col2 and net_col:
    sales2 = sales.loc[sale_mask].copy()
    sales2["month"] = pd.to_datetime(sales2[date_col2],errors="coerce").dt.to_period("M")
    ms = sales2.groupby("month")[net_col].sum().sort_index()
    ms.index = ms.index.astype(str)
    fig,ax = plt.subplots(figsize=(13,3.5))
    ms.plot(ax=ax,color=PALETTE[2],lw=2.5)
    ax.set_title("Monthly Sales Revenue (LKR)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x/1e6:.0f}M"))
    ax.tick_params(axis="x",rotation=45); plt.tight_layout(); plt.show()


## Tab 3 — Inventory Position (Module 3)

In [ ]:
try:
    inv_pos = load("m3_inventory_position.parquet")
    print(f"M3 inventory position: {len(inv_pos):,} rows")
    print("Columns:", inv_pos.columns.tolist())
    print(inv_pos.head(5).to_string())

    stock_col   = next((c for c in ["stock","on_hand","physical_stock"] if c in inv_pos.columns),None)
    demand_col  = next((c for c in ["demand","demand_lt","forecast_lt"] if c in inv_pos.columns),None)
    net_col2    = next((c for c in ["net_requirement","net_need"] if c in inv_pos.columns),None)

    if stock_col and demand_col:
        fig,axes = plt.subplots(1,2,figsize=(13,5))
        d = inv_pos[demand_col].clip(upper=inv_pos[demand_col].quantile(0.95))
        axes[0].scatter(d, inv_pos[stock_col].clip(upper=inv_pos[stock_col].quantile(0.95)),
                        alpha=0.25,s=12,color=PALETTE[0])
        axes[0].set_title("Inventory Position: Stock vs Lead-Time Demand")
        axes[0].set_xlabel("Lead-time demand"); axes[0].set_ylabel("Stock on hand")

        if net_col2:
            nc = inv_pos[net_col2].clip(-inv_pos[net_col2].abs().quantile(0.95),
                                         inv_pos[net_col2].abs().quantile(0.95))
            nc.hist(bins=60,ax=axes[1],color=PALETTE[3],edgecolor="white",alpha=0.8)
            axes[1].axvline(0,color=PALETTE[1],ls="--",lw=2,label="Balanced (stock=demand)")
            axes[1].set_title("Net Requirement Distribution
(negative=surplus, positive=shortage)")
            axes[1].set_xlabel("Net requirement (units)"); axes[1].legend()
        plt.tight_layout(); plt.show()
except FileNotFoundError:
    print("M3 inventory position not yet computed. Run Module 3 from the Pipeline page.")
